# Notebook 01 — Segmentação

**Projeto Final de Visão Computacional — Cenário C: Inspeção de Comprimidos**

Este notebook cumpre o **Bloco 2 da rubrica (0,5 ponto)**: isolar o comprimido do fundo
comparando **dois métodos de segmentação** (exigência do enunciado).

- **Método 1:** Limiarização de Otsu (tons de cinza)
- **Método 2:** Segmentação por cor/brilho no espaço HSV

Ambos passam pela mesma limpeza morfológica e extração do maior contorno.
As máscaras resultantes serão usadas no Notebook 02 para extrair as features.

> **Dataset:** OGYEIv2. As imagens ficam em `train/images`, `valid/images`, `test/images`,
> e a **classe está no nome do arquivo** (ex.: `no_spa_40_mg_s_004.jpg` → classe `no_spa_40_mg`).
> O sufixo `_s_`/`_u_` indica o tipo de iluminação (lateral/superior).

## 1. Configuração

Ajuste `DATASET_ROOT` para a pasta que contém `train`, `valid` e `test`. Liste em `CLASSES` os medicamentos escolhidos pela equipe.

In [4]:
import os
import sys
sys.path.append(os.path.abspath(".."))

import os
import glob
import re
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# caminho do dataset (ver config.py)
from config import DATASET_ROOT
# classes do projeto
CLASSES = [
    "algoflex_forte_dolo_400_mg",
    "apranax_550_mg",
    "donalgin_250_mg",
    "jutavit_c_vitamin"
]

OUT_DIR = "outputs/figuras"
os.makedirs(OUT_DIR, exist_ok=True)
np.random.seed(42)
print("Classes escolhidas:", CLASSES)

Classes escolhidas: ['algoflex_forte_dolo_400_mg', 'apranax_550_mg', 'donalgin_250_mg', 'jutavit_c_vitamin']


## 2. Leitura do dataset

A classe é extraída do nome do arquivo removendo o sufixo `_s_NNN`/`_u_NNN`.

In [5]:
def classe_do_arquivo(caminho):
    base = Path(caminho).stem
    return re.sub(r'_[su]_\d+$', '', base)

def listar_imagens(split):
    padrao = os.path.join(DATASET_ROOT, split, "images", "*.jpg")
    todas = glob.glob(padrao)
    return [p for p in todas if classe_do_arquivo(p) in CLASSES]

for split in ["train", "valid", "test"]:
    paths = listar_imagens(split)
    from collections import Counter
    cont = Counter(classe_do_arquivo(p) for p in paths)
    print(f"{split:6s}: {len(paths):3d} imagens  ->  {dict(cont)}")

train : 112 imagens  ->  {'algoflex_forte_dolo_400_mg': 28, 'apranax_550_mg': 28, 'donalgin_250_mg': 28, 'jutavit_c_vitamin': 28}
valid :  24 imagens  ->  {'algoflex_forte_dolo_400_mg': 6, 'apranax_550_mg': 6, 'donalgin_250_mg': 6, 'jutavit_c_vitamin': 6}
test  :  24 imagens  ->  {'algoflex_forte_dolo_400_mg': 6, 'apranax_550_mg': 6, 'donalgin_250_mg': 6, 'jutavit_c_vitamin': 6}


## 3. Os dois métodos de segmentação

Cada método devolve uma **máscara binária** (comprimido = branco, fundo = preto).
A função `limpar_mascara` é compartilhada: aplica abertura + fechamento morfológicos
para remover ruído e mantém apenas o **maior contorno** (o comprimido).

In [9]:
def redimensionar(bgr, max_dim=600):
    """reduz imagem grande"""
    h, w = bgr.shape[:2]
    escala = max_dim / max(h, w)
    if escala < 1:
        bgr = cv2.resize(bgr, (int(w*escala), int(h*escala)), interpolation=cv2.INTER_AREA)
    return bgr

def limpar_mascara(mask):
    """morfologia + maior contorno"""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(mask)
    if cnts:
        cv2.drawContours(out, [max(cnts, key=cv2.contourArea)], -1, 255, thickness=cv2.FILLED)
    return out

def segmentar_saturacao(bgr):
    """segmenta pelo canal de saturação"""
    bgr = redimensionar(bgr)
    s = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)[:, :, 1]
    _, th = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return limpar_mascara(th)

def segmentar_otsu(bgr):
    """Otsu com inversao de polaridade automatica"""
    bgr = redimensionar(bgr)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if (th > 0).sum() > (th == 0).sum():   # objeto deve ser a região menor
        th = cv2.bitwise_not(th)
    return limpar_mascara(th)


## 4. Comparação visual dos dois métodos

Mostramos, para uma amostra de cada classe, a imagem original e as máscaras de Otsu e HSV
lado a lado. Essa figura vai para o relatório (Bloco 2 pede *exemplos visuais e discussão*).

In [10]:
def mostrar_comparacao(paths, n=3):
    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))
    for i, p in enumerate(paths[:n]):
        
        bgr = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
        
        if bgr is None:
            print(f"Erro crítico: Não foi possível ler o arquivo no caminho: {p}")
            continue
            
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        m_otsu = segmentar_otsu(bgr)
        m_hsv  = segmentar_saturacao(bgr)
        axes[i,0].imshow(rgb);        axes[i,0].set_title(f"Original\n{classe_do_arquivo(p)}", fontsize=8)
        axes[i,1].imshow(m_otsu, cmap='gray'); axes[i,1].set_title("Método 1: Saturação", fontsize=8)
        axes[i,2].imshow(m_hsv, cmap='gray');  axes[i,2].set_title("Método 2: Otsu", fontsize=8)
        for j in range(3): axes[i,j].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "comparacao_segmentacao.png"), dpi=120, bbox_inches='tight')
    plt.show()

## 5. Avaliação quantitativa simples

Para discutir **acertos e falhas** (pedido pela rubrica), medimos quantas máscaras têm
área plausível (entre 3% e 80% da imagem). Máscaras vazias ou que tomam a imagem inteira
indicam falha de segmentação.

In [12]:
def fracao_plausivel(metodo, split="train"):
    paths = listar_imagens(split)
    ok = 0
    for p in paths:
        bgr = cv2.imdecode(np.fromfile(p, dtype=np.uint8), cv2.IMREAD_COLOR)
        
        if bgr is None:
            continue
            
        m = metodo(bgr)
        frac = (m > 0).mean()
        if 0.03 < frac < 0.80:
            ok += 1
    return ok, len(paths)

for nome, fn in [("Otsu", segmentar_otsu), ("HSV", segmentar_saturacao)]:
    ok, tot = fracao_plausivel(fn)
    print(f"{nome:5s}: {ok}/{tot} máscaras plausíveis ({100*ok/tot:.1f}%)")

Otsu : 58/112 máscaras plausíveis (51.8%)
HSV  : 0/112 máscaras plausíveis (0.0%)


## 6. Escolha do método e conclusão

Com base nos resultados acima e na inspeção visual, é escolhido **um** método como
principal para o Notebook 02 (normalmente o de maior taxa de máscaras plausíveis e bordas
mais limpas). Documente no relatório:

- qual método teve melhor desempenho e **por quê** (ex.: Otsu funciona bem com fundo
  uniforme; HSV ajuda quando há sombra lateral nas imagens `_s_`);
- exemplos de **falha** (sombra confundida com o comprimido, reflexo, etc.).

> A função escolhida (`segmentar_otsu` ou `segmentar_hsv`) será reaproveitada no Notebook 02.

In [13]:
# Defina aqui o método escolhido — será importado/reescrito no Notebook 02
METODO_ESCOLHIDO = segmentar_otsu   # ou segmentar_saturacao
print("Método principal definido:", METODO_ESCOLHIDO.__name__)

Método principal definido: segmentar_otsu
